# 05 · Leakage - how a model can look brilliant for the wrong reason

**Goal:** commit leakage on purpose, watch the scores become fantastic, and then learn the test that exposes it.

> **Leakage** = the model (or the data preparation) uses information that would not have been available at the forecast origin.

It is the single most common reason a forecasting result fails to survive contact with reality, and it is usually *accidental and invisible*. Below we create three leaks of increasing subtlety.
**Never do this in `src/`** - that is exactly why this lives in a tutorial.

In [1]:
from tutorial_utils import *
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
style()
clean = load_panel("clean")                      # clean scenario: nothing but leakage can fool us here
price = wide(clean, "wholesale_price_kes_kg"); lp = np.log(price)
h = 1

## The honest setup
Target: the log change of the price over the next week, `y = log P[t+1] − log P[t]`. Honest features use only data up to week `t`:
the change over the last week and 4 weeks, and the deviation from the 4-week average.

In [2]:
def honest_features(lp):
    return {"lr_1": lp - lp.shift(1), "lr_4": lp - lp.shift(4), "dev_mean_4": lp - lp.rolling(4).mean()}

def centred_leak(lp):        # subtle: a CENTRED window includes weeks t+1 and t+2
    return {"dev_centred_5": lp - lp.rolling(5, center=True, min_periods=3).mean()}

def lead_leak(lp):           # obvious: literally next week's price
    return {"next_week_change": lp.shift(-1) - lp}

y = lp.shift(-h) - lp

def to_frame(feats):
    X = pd.concat({k: v.stack() for k, v in feats.items()}, axis=1)
    X["y"] = y.stack(); X["ref"] = price.stack(); X["y_obs"] = price.shift(-h).stack()
    X.index.names = ["origin_week", "series"]
    return X.dropna().reset_index()

def evaluate(feats, label):
    d = to_frame(feats); cols = [c for c in d.columns if c not in ("origin_week", "series", "y", "ref", "y_obs")]
    tr = d[(d.origin_week >= 14) & (d.origin_week <= VAL_END - h)]
    te = d[(d.origin_week > VAL_END) & (d.origin_week <= TEST_END - h)]
    m = make_pipeline(StandardScaler(), Ridge(alpha=10)).fit(tr[cols], tr.y)
    pred = te.ref * np.exp(m.predict(te[cols]))
    naive = mae(te.y_obs, te.ref); model = mae(te.y_obs, pred)
    return dict(features=label, model_MAE=model, naive_MAE=naive, skill_vs_naive_pct=(1 - model / naive) * 100)

rows = [evaluate(honest_features(lp), "honest (lr_1, lr_4, dev_mean_4)"),
        evaluate({**honest_features(lp), **centred_leak(lp)}, "honest + centred 5-week window  (LEAK)"),
        evaluate({**honest_features(lp), **lead_leak(lp)}, "honest + next week's change  (LEAK)")]
pd.DataFrame(rows).set_index("features").round(2)

,model_MAE,naive_MAE,skill_vs_naive_pct
features,,,
"honest (lr_1, lr_4, dev_mean_4)",3.57,3.59,0.57
honest + centred 5-week window (LEAK),1.62,3.59,54.79
honest + next week's change (LEAK),0.02,3.59,99.56


**Look at the second row.** Replacing nothing, just *adding* a smoothed-looking feature, collapses the error. That is a feature that "knows" where the price is going because the centred window contains next week's and the week-after's price.
The third row is the cartoon version. A real leak is almost never labelled `next_week_change`; it is a rolling window with `center=True`, a scaler fitted on the whole file, an interpolation that uses later points,
or a join on a table that was updated after the origin.

## The test that exposes it: truncation invariance
**Idea.** If a feature is truly "as known at week t", then deleting all data after week `T0` must leave the feature values at origins ≤ `T0` **unchanged**.
Any change means the feature looked into the future.

In [3]:
def truncation_test(feature_fn, lp, cut=100):
    full, part = feature_fn(lp), feature_fn(lp.iloc[:cut])
    report = {}
    for k in full:
        a, b = full[k].loc[:cut].values, part[k].loc[:cut].values
        same = np.isclose(a, b, equal_nan=True).all(axis=1)           # per origin week
        report[k] = [int(w) for w in full[k].loc[:cut].index[~same]]
    return report

for name, fn in [("honest features", honest_features), ("centred window", centred_leak), ("next week's change", lead_leak)]:
    rep = truncation_test(fn, lp)
    verdict = "PASS" if not any(rep.values()) else "LEAK DETECTED"
    print(f"{name:20s} {verdict:14s} origins whose value changed when the future was deleted: {rep}")

honest features      PASS           origins whose value changed when the future was deleted: {'lr_1': [], 'lr_4': [], 'dev_mean_4': []}
centred window       LEAK DETECTED  origins whose value changed when the future was deleted: {'dev_centred_5': [99, 100]}
next week's change   LEAK DETECTED  origins whose value changed when the future was deleted: {'next_week_change': [100]}


The honest features pass; the leaky ones fail - and the test even tells you *where*: the centred window changes at the last two origins (it needs two future weeks), the lead feature at the last one.
This is precisely the test the pipeline runs (`tests/test_no_leakage.py`), on **every** feature of the real feature builder, for both the cleaned and the raw pipelines.

## A subtler leak: standardising with the whole file
Fitting a scaler (mean/variance) on all 156 weeks and then splitting leaks the *distribution* of the test period into the training features. Let us see whether it changes anything here.

In [4]:
def zscore_global(lp):
    f = honest_features(lp)
    return {k: (v - v.stack().mean()) / v.stack().std() for k, v in f.items()}          # uses ALL weeks
rows = [evaluate(honest_features(lp), "honest (scaler fitted on training rows only, inside the pipeline)"),
        evaluate(zscore_global(lp), "honest + features z-scored with whole-file mean/std")]
print(pd.DataFrame(rows).set_index("features").round(3).to_string())
rep = truncation_test(zscore_global, lp)
print("truncation test on the global z-score - number of origin weeks whose feature value changed when the future was deleted:",
      {k: len(v) for k, v in rep.items()}, "of 100")

                                                                   model_MAE  naive_MAE  skill_vs_naive_pct
features                                                                                                   
honest (scaler fitted on training rows only, inside the pipeline)      3.572      3.593               0.573
honest + features z-scored with whole-file mean/std                    3.572      3.593               0.573
truncation test on the global z-score - number of origin weeks whose feature value changed when the future was deleted: {'lr_1': 99, 'lr_4': 96, 'dev_mean_4': 97} of 100


The score barely moves (a global mean/std carries little information about next week's change), but the truncation test **does** flag it. A leak that is harmless today becomes harmful when
the test period is *different* - exactly the situation of this project (the test window contains new shocks). **Rule: anything learned from data (means, medians, scalers, imputers, thresholds, category lists)
must be learned from the training rows only, inside the modelling pipeline.** In the pipeline that is why imputation, scaling and one-hot encoding live inside scikit-learn `Pipeline`s that are re-fitted for every fold.

## Late-arriving data: leakage through the *calendar*
In the realistic and stressed scenarios, some indicator records reach the forecaster **one week after their period ends** (`report_delay_weeks = 1`). The value *belongs* to week `w`, but it is only *known* from week `w+1`.
Using it at origin `w` is a leak even though the number is "from the past".

In [5]:
rl = load_panel("realistic")
arr, dl = wide(rl, "market_arrivals_tonnes"), wide(rl, "report_delay_weeks")
print("share of arrival records that are late, by market:")
print((dl.T.groupby(lambda s: MARKETS[s[:6]]).mean().T.mean() * 100).round(1).to_string())

honest_lag0 = lambda a, d: a.where(d == 0)          # a late record is NOT visible at its own origin
leaky_lag0 = lambda a, d: a                          # ignores the delivery delay
i, j = [c for c in np.argwhere((dl.values == 1) & arr.notna().values) if c[0] >= 30][0]      # a late record from week 31 or later
week, series = dl.index[i], dl.columns[j]
arr2 = arr.copy(); arr2.iloc[i, j] = arr.iloc[i, j] * 100                         # "what if that late value had been 100x larger?"
print(f"\nperturbing the LATE arrival record of {series}, week {week}:")
for name, fn in [("honest as-known lag-0", honest_lag0), ("leaky lag-0", leaky_lag0)]:
    b, a = fn(arr, dl).iloc[i, j], fn(arr2, dl).iloc[i, j]
    print(f"  {name:24s} feature at origin {week}: before = {b}, after = {a}   -> {'CHANGED (leak)' if not (np.isnan(b) and np.isnan(a)) and b != a else 'unchanged (not visible yet)'}")
b, a = arr.shift(1).iloc[i + 1, j], arr2.shift(1).iloc[i + 1, j]
print(f"  one week later (lag-1 at origin {week + 1}): before = {b:.1f}, after = {a:.1f}  -> the value is legitimately available now")

share of arrival records that are late, by market:
series
Eldoret     0.0
Kisumu     30.9
Mombasa    28.4
Nairobi     0.0
Nakuru      0.0

perturbing the LATE arrival record of MKT-04|CRP-04, week 31:
  honest as-known lag-0    feature at origin 31: before = nan, after = nan   -> unchanged (not visible yet)
  leaky lag-0              feature at origin 31: before = 247.095, after = 24709.5   -> CHANGED (leak)
  one week later (lag-1 at origin 32): before = 247.1, after = 24709.5  -> the value is legitimately available now


This "perturbation test" is the second leakage test in the pipeline: change something the model should *not* be able to see, and verify that its inputs do not change.

## Audit columns are not features - and labels are not tools
The data contain columns that exist only because we *generated* the data: `latent_price_kes_kg`, `data_quality_flag`, `event_ids`, `event_affected`. They are for grading and auditing.
Consider "cleaning" the reported prices by dropping the rows that `data_quality_flag` calls `suspect`:

In [6]:
raw = wide(rl, "wholesale_price_kes_kg")
suspect = wide(rl, "data_quality_flag").eq("suspect")
accepted, _ = clean_price_wide(raw)
ref_causal = accepted.ffill(limit=8)
ref_oracle = raw.where(~suspect).ffill(limit=8)           # cheating: uses the audit label
def naive_mae_window(ref, lo, hi):
    out = []
    for hh in HORIZONS:
        o = np.arange(lo, eval_hi(hh, hi) + 1)
        t = raw.shift(-hh).loc[o]; ok = t.notna() & ~suspect.shift(-hh, fill_value=False).loc[o] & ref_causal.loc[o].notna()
        out.append((t - ref.loc[o]).abs().where(ok).stack().mean())
    return np.mean(out)
for label, lo, hi in [("test window (origins 131+)", VAL_END + 1, TEST_END), ("whole period (origins 14+)", 14, TEST_END)]:
    print(f"naive MAE, {label}:  causal cleaning {naive_mae_window(ref_causal, lo, hi):.3f}   |   label 'cleaning' {naive_mae_window(ref_oracle, lo, hi):.3f}")

naive MAE, test window (origins 131+):  causal cleaning 6.165   |   label 'cleaning' 6.165
naive MAE, whole period (origins 14+):  causal cleaning 4.807   |   label 'cleaning' 4.807


The gap between the two lines is what knowing the labels would be worth - it is the performance you would *imagine* having, but could not get in real life. (Here the causal filter is good enough that the gap is tiny:
it misses one error in the first weeks, where it has too little history, and briefly distrusts one genuine price crash - see notebook 03.) The point is the principle: a cheat that happens to be harmless today hides how your cleaning
would really perform on messier data. The pipeline uses the labels in exactly one place:
to **exclude corrupted target rows from scoring** (we do not want to grade a model against a typo). They never enter a model input, and a unit test fails if a column named `latent…` or `y_…` appears in any feature set.

## Leakage checklist (print this)
1. Does every feature depend only on weeks ≤ origin **and** on records delivered by the origin? → *truncation* and *perturbation* tests.
2. Is every learned statistic (mean, median, scaler, encoder, threshold) fitted on training rows only? → keep them inside Pipelines.
3. Do training rows' **targets** lie inside the training window? (notebook 06 - "purging")
4. Does cleaning use only the past? → causal filter, no centred windows, no `interpolate()` across the origin.
5. Are audit labels used only for grading? → never as inputs.
6. Is the test set touched once, after all choices were made? (notebook 06)

## Exercises
1. Make `centred_leak` use `center=True` with window 3 instead of 5. Does the skill change? At which origins does the truncation test fail now?
2. Write a leaky feature of your own: e.g. `lp.interpolate()` applied to the realistic panel (it fills gaps using *later* values). Use the truncation test to catch it.
3. Add a **honest** feature of your choice (for example the 8-week change) and confirm it passes both tests.
4. Run the perturbation test on the transport-cost indicator (`transport_cost_kes_tonne`, also delayed in the real data).

**Where this lives in the pipeline:** `src/agriprice/features.py` (`asof_lag`, the as-known contract in the module docstring), `docs/forecast_time_data_contract.md`, and the tests in `tests/test_no_leakage.py`.